# S3 · Range diagnostics on real sCTs (go/no-go 1)

Week 3 asks one question, fixed in advance in `configs/s3_preregistration.json`:

> Do out-of-fold sCT errors on SynthRAD2025 development cases carry excess low-frequency energy **along the beam**,
> relative to the same errors shuffled?

If they do, the range-aware loss of weeks 4–5 has something to act on (docs/theory.md, Result A and Proposition 2).
If they do not, range error is dominated by white or interface errors that voxel losses already see, and the paper
re-centres on Results B–C (projection consistency, tails, proper evaluation), which do not depend on this outcome.

Pipeline:

1. verify the frozen split and the pre-registration;
2. train 2.5D U-Nets with two point losses, **cross-fitted** on the development pool so that every diagnosed prediction
   is out of fold; test cases are never loaded;
3. per case and source: range metrics with permutation controls, and tapered periodograms of the SPR error along and
   across beams, relative to shuffled errors;
4. apply the pre-registered rule, then the secondary tests.

Sources: the bulk-density baseline, `l1_hu` (L1 in HU, the usual sCT loss, Bayes act = median) and `l2_spr` (L2 in SPR,
Bayes act for squared range error, Proposition 4).

Environment variables: `SYNTHRAD2025_ROOT` (dataset), `SCT_RUNS_ROOT` (training runs, default `outputs/s3/runs`),
`SCT_CACHE_DIR` (default `outputs/s3/cache`), `SCT_TASK` (`1` for MRI, `2` for CBCT), `SCT_TRAIN_IN_NOTEBOOK=1` to train
here instead of with `sctrange-train`. Without the dataset, a synthetic fixture is generated: its MR is a surrogate
function of the CT, so **fixture numbers validate the pipeline only and carry no scientific meaning.**

The notebook runs the code of this repository (`src/`), whether or not the package is installed.

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own (Colab, another machine), it first downloads the repository from GitHub (no git needed;
`SCTRANGE_REF` selects a branch, tag or commit, default `main`; `SCTRANGE_HOME` the destination), then installs it.
pip only runs when `sctrange` is missing or installed from elsewhere; `SCTRANGE_REINSTALL=1` forces it.

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", Path.cwd() / "sct-range")).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"], check=True)
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

In [ ]:
import json
import os
import warnings
from dataclasses import asdict, replace
from pathlib import Path

warnings.filterwarnings("ignore", message=r"`torch\.jit\.script` is deprecated")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sctrange.cache import VolumeCache
from sctrange.data import bulk_density_sct, discover_cases, load_case, read_volume
from sctrange.diagnostics import Preregistration, decide, diagnose_case, paired_test
from sctrange.evaluation import DiagnosticConfig
from sctrange.hlut import HLUT
from sctrange.phantoms import write_fixture_dataset
from sctrange.plotting import apply_style, plot_beam_spectra, plot_signed_bias, plot_training_curves
from sctrange.split import SplitSpec, load_split, split_hash, stratified_split, write_split
from sctrange.training import TrainConfig, crossfit

In [ ]:
DATA_ROOT = Path(os.environ.get("SYNTHRAD2025_ROOT", REPO_ROOT / "data" / "synthrad2025"))
OUTPUT_DIR = REPO_ROOT / "outputs" / "s3"
FIGURE_DIR = OUTPUT_DIR / "figures"
RUNS_ROOT = Path(os.environ.get("SCT_RUNS_ROOT", OUTPUT_DIR / "runs"))
CACHE_DIR = Path(os.environ.get("SCT_CACHE_DIR", OUTPUT_DIR / "cache"))
FROZEN_SPLIT_PATH = REPO_ROOT / "splits" / "synthrad2025_local_test.json"
PREREGISTRATION_PATH = REPO_ROOT / "configs" / "s3_preregistration.json"
TASK = os.environ.get("SCT_TASK", "1")

SEED = 20261013
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LOSSES = ("l1_hu", "l2_spr")
N_FOLDS = 5
DIAGNOSTIC = DiagnosticConfig(angles_deg=tuple(range(0, 360, 45)), target_radius_mm=25.0, seed=SEED, device=DEVICE)
MAX_CASES = None
hlut = HLUT.pyradplan_default()

USE_FIXTURE = not DATA_ROOT.exists() or next(DATA_ROOT.rglob("ct.mha"), None) is None
if USE_FIXTURE:
    DATA_ROOT = OUTPUT_DIR / "fixture" / "data"
    if next(DATA_ROOT.rglob("ct.mha"), None) is None:
        write_fixture_dataset(DATA_ROOT, centers=("A", "B", "C"), cases_per_stratum=5, seed=SEED)
    N_FOLDS = 3
TRAIN_IN_NOTEBOOK = USE_FIXTURE or os.environ.get("SCT_TRAIN_IN_NOTEBOOK") == "1"
SPLIT_PATH = OUTPUT_DIR / "fixture_split.json" if USE_FIXTURE else FROZEN_SPLIT_PATH

TRAIN_CONFIG = TrainConfig(seed=SEED, device=DEVICE)
if USE_FIXTURE:
    TRAIN_CONFIG = replace(TRAIN_CONFIG, context=1, patch=96, batch_size=8, steps=300, warmup_steps=30,
                           base_channels=16, depth=3, log_every=25, samples_per_case=2000)

torch.manual_seed(SEED)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
apply_style()
pd.set_option("display.width", 160)
shown = DATA_ROOT.resolve().relative_to(REPO_ROOT.resolve()).as_posix() if DATA_ROOT.resolve().is_relative_to(REPO_ROOT.resolve()) else DATA_ROOT
print(f"data: {shown}{'  [synthetic fixture]' if USE_FIXTURE else ''}")
print(f"task {TASK} · device {DEVICE} · folds {N_FOLDS} · train in notebook: {TRAIN_IN_NOTEBOOK}")

## 1 · Frozen split and pre-registration

The split must already exist on real data (week 1) and is only verified here. The pre-registration file is hashed and
the hash goes into the report, so the decision rule demonstrably predates the results.

In [ ]:
records = discover_cases(DATA_ROOT)
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
elif USE_FIXTURE:
    write_split(stratified_split(records, SplitSpec(seed=SEED)), SPLIT_PATH)
    split = load_split(SPLIT_PATH)
else:
    raise FileNotFoundError(f"{SPLIT_PATH} is missing: freeze the split first (week 1)")

development_ids = set(split["development"])
development = [r for r in records if r.case_id in development_ids and r.task == TASK][:MAX_CASES]
assert not development_ids & set(split["test"])

prereg = Preregistration.load(PREREGISTRATION_PATH)
display(pd.Series({"split sha256": split_hash(split), "development cases (this task)": len(development),
                   "test cases (never loaded)": len(split["test"]),
                   "pre-registration sha256": prereg.fingerprint}, name="value").to_frame())
display(pd.json_normalize(prereg.raw, sep=" · ").T.rename(columns={0: "value"}))

**Rule.** For the primary source (`l1_hu`), over development cases:

| Verdict | Condition |
|---|---|
| GO | median along-beam low-frequency excess ≥ 1.5 **and** median target-segment global-shuffle ratio ≥ 1.2 |
| WEAK | median along-beam low-frequency excess ≥ 1.15 |
| NO-GO | otherwise |

*Low-frequency excess* is the share of error power below 0.05 cycles/mm (periods above 20 mm, organ scale) in 64 mm
windows along the beam (beams at 0°, 60° and 120°), divided by the same share for the errors shuffled over the body;
1 means white-like layout.
The *global-shuffle ratio* is the RMS range error to the distal target edge divided by that of the shuffled errors at
identical MAE. A uniform bias leaves both statistics at 1 by construction (the shuffle preserves the mean); it is
reported separately as the signed range error.

## 2 · Cross-fitted training

Development cases are cached once (normalised, body-cropped, memory-mapped), then split into stratified folds. For each
loss and fold, a model is trained on the other folds and predicts the held-out fold. Interrupted runs resume from the
saved fold checkpoints. On real data, train with the CLI on a GPU and rerun this notebook:

```bash
for loss in l1_hu l2_spr; do
  sctrange-train --data-root "$SYNTHRAD2025_ROOT" --split splits/synthrad2025_local_test.json \
    --cache-dir outputs/s3/cache --out outputs/s3/runs --loss $loss --task 1 --folds 5
done
```

In [ ]:
cache = VolumeCache(CACHE_DIR)
print(f"{cache.build(development)} new cases cached")

histories = {}
for loss in LOSSES:
    run_dir = RUNS_ROOT / loss
    if TRAIN_IN_NOTEBOOK:
        crossfit(development, cache, replace(TRAIN_CONFIG, loss=loss), hlut, N_FOLDS, run_dir)
    elif not (run_dir / "predictions").exists():
        print(f"{loss}: no predictions under {run_dir}; train with sctrange-train first")
        continue
    if (run_dir / "history.csv").exists():
        histories[loss] = pd.read_csv(run_dir / "history.csv")

display(pd.Series(asdict(TRAIN_CONFIG), name="value").to_frame())
if histories:
    plot_training_curves(histories).savefig(FIGURE_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()
    display(pd.concat({loss: h.groupby("fold").tail(1).set_index("fold")[["step", "loss", "seconds"]]
                       for loss, h in histories.items()}).round(3))

## 3 · Per-case diagnostics

For every development case: the bulk-density baseline and every available out-of-fold prediction are compared with the
reference CT. Range metrics use 8 coplanar beams and a 25 mm surrogate target at the body centroid (SynthRAD2025
releases no PTV). Spectra use 64 mm windows lying entirely inside the body, on three beams 60° apart (0°, 60°, 120°):
with beams every 45°, the across direction of one beam is the along direction of another and the two spectra would be
identical by construction.

In [ ]:
rows, spectra = [], {}
for record in development:
    volumes = load_case(record, ("ct", "mask"))
    ct, mask, spacing = volumes["ct"].array, volumes["mask"].array, volumes["ct"].spacing
    sources = {"bulk density": bulk_density_sct(mask)}
    for loss in LOSSES:
        path = RUNS_ROOT / loss / "predictions" / f"{record.case_id}.mha"
        if path.exists():
            sources[loss] = read_volume(path).array
    for source, sct in sources.items():
        metrics, case_spectra = diagnose_case(ct, mask, sct, spacing, hlut, DIAGNOSTIC, prereg)
        rows.append({"case_id": record.case_id, "region": record.region, "center": record.center,
                     "source": source, **metrics})
        spectra.setdefault(source, []).append(case_spectra)

diagnostics = pd.DataFrame(rows)
diagnostics.to_csv(OUTPUT_DIR / "s3_diagnostics.csv", index=False)
print(f"{diagnostics['case_id'].nunique()} cases, {len(diagnostics)} evaluations, sources: {sorted(spectra)}")

columns = ["mae_hu", "target_mean_signed_mm", "target_rms_mm", "target_cvar_mm", "target_coherence",
           "target_intra_ratio", "target_global_ratio", "lf_excess_along", "lf_excess_across"]
display(diagnostics.groupby(["source", "region"])[columns].median().round(3))

Mean error power, as a share of total power, relative to the same share for shuffled errors. Values above 1 at low
frequency (grey band) mean coherent, organ-scale errors; a curve that rises along the beam but not across it would point
to beam-direction structure, a flat one to isotropic coherence.

In [ ]:
plot_beam_spectra(spectra, prereg.cutoff).savefig(FIGURE_DIR / "beam_spectra.png", bbox_inches="tight")
plt.show()
plot_signed_bias(diagnostics).savefig(FIGURE_DIR / "signed_bias.png", bbox_inches="tight")
plt.show()

## 4 · Pre-registered decision and secondary tests

Secondary tests are reported whatever the verdict and do not change it:

- **Proposition 5 on real data**: in the abdomen (uncertain gas), the signed range error of `l1_hu` exceeds that of
  `l2_spr` (paired one-sided Wilcoxon over cases);
- **tails**: target CVaR90 of `l1_hu` versus `l2_spr` (two-sided);
- **image quality**: MAE of `l1_hu` versus `l2_spr` (two-sided), to read the two previous results at comparable MAE.

In [ ]:
decision = decide(diagnostics, prereg)
secondary = prereg.raw["secondary"]
first, second = secondary["bias_pair"]
tests = {
    "signed bias, AB (l1_hu − l2_spr > 0)": paired_test(diagnostics, secondary["bias_column"], first, second,
                                                       secondary["bias_region"], secondary["bias_alternative"]),
    "target CVaR90 (l1_hu − l2_spr)": paired_test(diagnostics, secondary["tail_column"], first, second),
    "MAE (l1_hu − l2_spr)": paired_test(diagnostics, "mae_hu", first, second),
}
display(pd.Series(decision, name="value").to_frame())
display(pd.DataFrame(tests).T.round(4))
if USE_FIXTURE:
    print("Fixture run: the verdict exercises the decision code only.")

## 5 · Report

In [ ]:
report = {
    "fixture": USE_FIXTURE,
    "task": TASK,
    "split_sha256": split_hash(split),
    "train_config": asdict(TRAIN_CONFIG),
    "folds": N_FOLDS,
    "decision": decision,
    "secondary": tests,
    "medians_by_source": diagnostics.groupby("source")[columns].median().to_dict(orient="index"),
}
(OUTPUT_DIR / "s3_report.json").write_text(json.dumps(report, indent=2, default=float))
status = pd.DataFrame([
    ("Frozen split verified", split_hash(split)[:12], True),
    ("Pre-registration hashed", prereg.fingerprint[:12], True),
    ("Out-of-fold predictions", ", ".join(s for s in spectra if s != "bulk density") or "none",
     all(loss in spectra for loss in LOSSES)),
    ("Diagnostics", f"{len(diagnostics)} evaluations", len(diagnostics) > 0),
    ("Verdict on real data", decision["verdict"], not USE_FIXTURE and decision["verdict"] != "NOT EVALUATED"),
], columns=["item", "detail", "passed"])
display(status)

## What each verdict implies for weeks 4–5

| Verdict | Weeks 4–5 |
|---|---|
| GO | Run arms R1–R6 as planned. The range loss (R5) and the VaR/ES head (R6) are tested against R3 (L2 in SPR) and R4 (R3 + per-tissue bias correction) on CVaR at matched MAE. |
| WEAK | Keep R1–R4 and R6; run R5 with a reduced sweep. Report the effect size of coherence rather than a headline gain. |
| NO-GO | Drop R5 as a contribution. The paper rests on Results B–C: estimator choice (R1 vs R3), tails (R6 and the generative arm), projected CRPS. |

In every case, the secondary bias test decides whether the L1-versus-SPR-mean result of Proposition 5 holds on patients,
which is reportable on its own.

Weeks 4–5 run in `notebooks/s4_regression_arms.ipynb`, with their own pre-registered rule (`configs/s4_preregistration.json`).